# 02. Filmreviews mit k nächsten Nachbarn klassifizieren

**Leitfrage:** Welche bekannten Filmreviews stimmen über das Urteil zu einer neuen Review ab?

**Arbeitsauftrag:** [kNN trainieren und vorhersagen](#classify) · [Die drei Stimmen erklären](#neighbors)

In [ ]:
%pip install scikit-learn==1.9.1

## 1. Acht Trainingsreviews und zwei neue Texte

Im letzten Notebook hat die nächste Review ihr Label an den neuen Text weitergegeben. **k-Nearest Neighbors (kNN)** lässt die `k` nächsten Trainingsreviews abstimmen. Die **Distanz** zwischen zwei Wortvektoren bestimmt ihre Nähe: Je kleiner sie ist, desto ähnlicher sind die Reviews nach diesem Maß.

Wir wählen `k = 3`. Jede der drei Reviews hat eine Stimme. Tragen zum Beispiel zwei Nachbarn das Label `neg` und einer das Label `pos`, lautet die Vorhersage `neg`. [Grundidee der kNN-Klassifikation](https://scikit-learn.org/stable/modules/neighbors.html#nearest-neighbors-classification)

### 1.1 Die Kurzreviews lesen

In der Trainingsliste folgt auf jeden Filmkommentar seine bekannte Bewertung: `pos` für positiv, `neg` für negativ.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

reviews = [
    ('Great acting and warm humor.', 'pos'),
    ('Beautiful original music and warm dialogue.', 'pos'),
    ('Sharp dialogue and great acting.', 'pos'),
    ('Strong moving story with beautiful music.', 'pos'),
    ('Flat acting and dull humor.', 'neg'),
    ('Weak story with dull dialogue.', 'neg'),
    ('Slow pacing and flat music.', 'neg'),
    ('Weak uneven acting with slow pacing.', 'neg'),
]

for i, (text, label) in enumerate(reviews, start=1):
    print(i, label, text)

In [ ]:
new_texts = ['Great acting and dull humor.', 'Beautiful music and flat acting.']
print('New reviews:', *new_texts, sep='\n')

### 1.2 Wortvektoren vorbereiten

Gib für beide neuen Texte einen Tipp ab. Bereite anschließend wie in [Notebook 00](00_count_vectorizer.ipynb) die Wortvektoren vor. Jede Zeile steht für eine Review, jede Spalte für ein Wort aus den Trainingsreviews.

In [ ]:
train_texts = [text for text, label in reviews]
labels = [label for text, label in reviews]

vectorizer = CountVectorizer()
known_vectors = vectorizer.fit_transform(train_texts)

print('Training matrix:', known_vectors.shape)
print(vectorizer.get_feature_names_out())

In [ ]:
new_vectors = vectorizer.transform(new_texts)
print('Vectorized new text:', *(new_vectors.toarray().tolist()), sep='\n')

Die Spaltenüberschriften gelten auch für die beiden neuen Zeilen. Welches Wort gehört zur ersten Spalte? Lies ab, wie oft es in jedem neuen Text vorkommt.

<a id="classify"></a>

## 2. Zwei Kurzreviews klassifizieren

### 2.1 Die Trainingsbeispiele hinterlegen

Erzeuge mit `KNeighborsClassifier` ein Modell namens `model`. Setze `n_neighbors=3` für drei Stimmen und `metric='cosine'` für die Kosinusdistanz. Rufe danach `fit` mit den Trainingsvektoren `known_vectors` und ihren `labels` auf. Damit stehen die acht beschrifteten Reviews als Nachbarn zur Verfügung. [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) · [fit](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier.fit)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = ...  # TODO: Create model with three neighbors and cosine distance.
model.fit(...) # TODO: Fit model with known_vectors and labels.

### 2.2 Die Labels vorhersagen

Rufe `predict` für die beiden neuen Wortvektoren auf. Speichere die Labels in `predictions`. Ihre Reihenfolge entspricht den Zeilen von `new_vectors`. [predict](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier.predict)

In [ ]:
predictions = ...  # TODO: Predict both labels for new_vectors and store them in predictions.

In [ ]:
assert len(predictions) == len(new_texts)
for text, label in zip(new_texts, predictions):
    print(text, label)

<a id="neighbors"></a>

### 2.3 Wer hat für die erste Review abgestimmt?

`kneighbors` liefert die Abstände und die Zeilennummern der drei nächsten Trainingsreviews. Mit `new_vectors[:1]` fragen wir nach den Nachbarn des ersten neuen Texts. [kneighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier.kneighbors)

In [ ]:
distances, indices = model.kneighbors(new_vectors[:1])
print('Query:', new_texts[0])
for distance, index in zip(distances[0], indices[0]):
    print(f'{distance:.3f}', labels[index], train_texts[index])

Zähle die positiven und negativen Stimmen. Erkläre die Vorhersage der ersten Review mithilfe dieser Mehrheit. Welche Wörter verbinden sie mit ihren Nachbarn?